This notebook is used to synthesize the trained models with HLS4MLs backend Vitis Unified. Some synthesis include bitfile-generation, and varies with strategy.

In [1]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.metrics import accuracy_score

# Load Vitis into path
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

I0000 00:00:1778155838.784231 1316163 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
model_to_test = 'jettag-hgq2'

model_configs = [
    # AdaptiveHP-models
#    {
#        "description": "AdaptiveHP acc=0.7117 ebops=303 Distributed Arithmetic",
#        "model_revision": "Training_AdaptiveHP",
#        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7117_ebops=303.keras",
#        "hls4ml_strategy": "DA",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "acc=0.7117_ebops=303_VU_DA_bitfile",
#    },
#    {
#        "description": "AdaptiveHP acc=0.7117 ebops=303 latency",
#        "model_revision": "Training_AdaptiveHP",
#        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7117_ebops=303.keras",
#        "hls4ml_strategy": "latency",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "acc=0.7117_ebops=303_VU_latency_bitfile",
#    },
#
#    {
#        "description": "AdaptiveHP acc=0.7426 ebops=1001 Distributed Arithmetic",
#        "model_revision": "Training_AdaptiveHP",
#        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7426_ebops=1001.keras",
#        "hls4ml_strategy": "DA",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "acc=0.7426_ebops=1001_VU_DA_bitfile",
#    },
#    {
#        "description": "AdaptiveHP acc=0.7512 ebops=2895 Distributed Arithmetic",
#        "model_revision": "Training_AdaptiveHP",
#        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7512_ebops=2895.keras",
#        "hls4ml_strategy": "DA",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "acc=0.7512_ebops=2895_VU_DA_bitfile",
#    },
#    # FixedHP-models
#    {
#        "description": "FixedHP acc=0.7590 ebops=11634 Distributed Arithmetic",
#        "model_revision": "Training_FixedHP",
#        "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7590_ebops=11634.keras",
#        "hls4ml_strategy": "DA",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "acc=0.7590_ebops=11634_VU_DA_bitfile",
#    },

    {
        "description": "FixedHP acc=0.7659 ebops=21624 Distributed Arithmetic",
        "model_revision": "Training_FixedHP",
        "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7659_ebops=21624.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7659_ebops=21624_VU_DA_bitfile",
    },
    {
        "description": "FixedHP acc=0.7659 ebops=21624 latency",
        "model_revision": "Training_FixedHP",
        "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7659_ebops=21624.keras",
        "hls4ml_strategy": "latency",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7659_ebops=21624_VU_latency_bitfile",
    },
]

In [3]:
# Load dataset which is preprocessed in another notebook
#X_train_val = np.load('Data/x_train_val.npy')
#X_test = np.load('Data/x_test.npy')
#y_train_val = np.load('Data/y_train_val.npy')
#y_test = np.load('Data/y_test.npy')
#classes = np.load('Data/classes.npy', allow_pickle=True)


In [4]:
import os

def prepare_directory(model_config):
    output_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
    )
    os.makedirs(output_dir, exist_ok=True)

    description = f"""
    Description of HLS4ML-project.

    {model_config['description']}

    - Bitfile: {model_config['hls4ml_generate_bitfile']}
    - Environment: devenv-vu-hgq+da (environment-HGQ+DA.yml)
    - Target Device: KV260 (xck26-sfvc784-2LV-c)
    - Dataset: HLS4ML LHC Jets
    - Vivado/Vitis: 2025.2
    - Model Architecture: {model_to_test}
    - Model Revision: {model_config['model_revision']}
    - HLS4ML Revision: {model_config['hls4ml_revision']}

    The model summary is in the parent-directory, `summary.txt`
    """
    with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
        f.write(description)

    return output_dir

In [5]:
from keras.models import load_model
import hgq.layers
import hls4ml

def compile_model(keras_model_path, output_dir, hls4ml_strategy):
    model = load_model(keras_model_path)
    
    hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')
    
    strategy = 'Distributed Arithmetic' if hls4ml_strategy == 'DA' else hls4ml_strategy
    hls_config['Model']['Strategy'] = strategy # https://fastmachinelearning.org/hls4ml/api/configuration.html#top-level-configuration

    hls_model = hls4ml.converters.convert_from_keras_model( 
        model,    
        backend     =   'vitisunified',
        hls_config  =   hls_config,
        output_dir  =   output_dir, 
        board       =   'kv260',
        part        =   'xck26-sfvc784-2LV-c',
        clock_period=   '5',
    )
    hls_model.compile()
    return hls_model

In [ ]:
# Hotfix for crashing, see README
os.environ['LD_PRELOAD'] = '/lib/x86_64-linux-gnu/libudev.so.1'

process_model_durations = []

# Run through every model
for i, model_config in enumerate(model_configs):
    print(f"Processing model {i+1}/{len(model_configs)}: {model_config['description']}")
    print(f"%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
    start_time = time.time()
    output_dir = prepare_directory(model_config)
    
    hls_model = compile_model(
        model_config["keras_model_path"],
        output_dir,
        model_config["hls4ml_strategy"],
    )

    # Create complete bitfile (Vitis Unified-backend) or IP-block (Vitis-backend)
    hls_model.build(
        synth=True,
        bitfile=model_config["hls4ml_generate_bitfile"],
        csim=False # Simulation (CSIM and COSIM) needs input_data_tb and output_data_tb https://fastmachinelearning.org/hls4ml/autodoc/hls4ml.converters.html#hls4ml.converters.convert_from_keras_model
    )
    
    elapsed_time = time.time() - start_time
    process_model_durations.append({
        'model': model_config['description'],
        'time_seconds': elapsed_time,
        'time_minutes': elapsed_time / 60
    })
    print(f"\nModel {i+1} completed in {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")


Processing model 1/2: FixedHP acc=0.7659 ebops=21624 Distributed Arithmetic
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Thu May  7 12:10:45 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
INFO: [HLS 200-2005] Using work_dir /work/development/jettag/jettag-hgq2/Training_FixedHP/hls4ml_prj_acc=0.7659_ebops=21624_VU_DA_bitfile/vitis_workspace/myproject/vitis_unified_project 
INFO: [HLS 200-2176] Writing Vitis IDE component file /work/development/jettag/jettag-hgq2/Training_FixedHP/hls4ml_prj_acc=0.7659_ebops=21624_VU_DA_bitfile/vitis_workspace/myproject/vitis_unified_project/vitis-comp.json
INFO: [HLS 200-10] Creating and opening component '/work/development/jettag/jettag-hgq2/Training_FixedHP/hls4ml_prj_acc=0.7659_ebops=21624_VU_DA_bitfile/vitis_workspace/mypro

In [ ]:
import pandas as pd

timing_df = pd.DataFrame(process_model_durations)
timing_df = timing_df[["model", "time_seconds", "time_minutes"]]
timing_df.to_csv("timing_summary.csv", index=False)

display(timing_df)

total_time = timing_df["time_seconds"].sum()
average_time = timing_df["time_seconds"].mean()
print(f"\nTotal time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"Average time per model: {average_time:.2f} seconds ({average_time/60:.2f} minutes)")

,model,time_seconds,time_minutes
0,AdaptiveHP acc=0.7117 ebops=303 Distributed Ar...,357.863168,5.964386



Total time: 357.86 seconds (5.96 minutes)
Average time per model: 357.86 seconds (5.96 minutes)


Copy bitfile and hwh from export-directory to directory for onboard-verification.
Make sure the driver and testdata is copied manually.

In [ ]:
import glob
import shutil

target_dir = '../../onboard-verification/jettag/DUT'

for model_config in model_configs:
    export_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
        'export'
    )
    model_target_dir = os.path.join(
        target_dir,
        f"{model_config['model_revision']}_{model_config['hls4ml_revision']}"
    )
    os.makedirs(model_target_dir, exist_ok=True)

    for extension in ('bit', 'hwh'):
        matches = sorted(glob.glob(os.path.join(export_dir, f'*.{extension}')))
        if not matches:
            print(f"ERROR: No .{extension} file found in {export_dir}")
            continue
        shutil.copy2(matches[0], model_target_dir)
        print(f"Copied {matches[0]} -> {model_target_dir}")

Copied /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7117_ebops=303_VU_DA_bitfile/export/system.bit -> ../../onboard-verification/jettag/DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_DA_bitfile
Copied /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7117_ebops=303_VU_DA_bitfile/export/system.hwh -> ../../onboard-verification/jettag/DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_DA_bitfile
Copied /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7117_ebops=303_VU_latency_bitfile/export/system.bit -> ../../onboard-verification/jettag/DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_latency_bitfile
Copied /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7117_ebops=303_VU_latency_bitfile/export/system.hwh -> ../../onboard-veri